# SQL / SPARQL Performance: Tom Holland -> Brad Pitt

This notebook runs three queries against the project's local services and compares
their execution times: movies longer than 90 minutes, a path with **one intermediary**,
and a path with **three intermediaries**. Saved outputs come from real executions;
select **Run All** to repeat the measurements.

Prerequisites: start the services with `docker compose up --build --wait`, then
select a Python 3 kernel in Jupyter or VS Code. SPARQL mode uses the Python standard
library and IPython provided by Jupyter. SQL comparison also requires `psycopg2`,
which the project already uses. If needed, install it in the kernel with
`%pip install psycopg2-binary`. All execution code and SQL/SPARQL query definitions are in the notebook cells.

Run All refreshes the measurements and displays the results directly in the notebook.

## 1. Configuration

Set `COMPARE_SQL = False` to measure SPARQL only. The first execution is measured
separately, followed by five repetitions. Connections are established before timing;
caches are not cleared.

In [1]:
import csv
import hashlib
import http.client
import json
import os
import platform
import statistics
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, urlsplit
from IPython.display import Code, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'submit' / 'Database' / 'show.csv').exists())
HERE = ROOT / 'new_features'
REPEATS = 5
COMPARE_SQL = True
SPARQL_ENDPOINT = os.environ.get('SPARQL_ENDPOINT', 'http://127.0.0.1:3030/ds/query')
PG_CONFIG = {
    'host': os.environ.get('PGHOST', '127.0.0.1'),
    'port': os.environ.get('PGPORT', '5432'),
    'dbname': os.environ.get('PGDATABASE', 'show_catalog'),
    'user': os.environ.get('PGUSER', 'postgres'),
    'password': os.environ.get('PGPASSWORD', 'show_catalog_local'),
}
assert REPEATS >= 1
print(f'Project: {ROOT.name} ; repetitions: {REPEATS} ; SQL comparison: {COMPARE_SQL}')

Project: KnowledgeandReasoning ; repetitions: 5 ; SQL comparison: True


## 2. Inline SQL and SPARQL queries

All queries are defined directly in the following cells. Edit these definitions to
change the benchmark; the notebook does not load `.sql` or `.rq` files.
Both versions use the same filters and a limit of 20 results. The search follows
**actor–film** links without the precomputed `knows` shortcut.


In [2]:
CASES = {
    'movies_over_90': 'Movies longer than 90 minutes',
    'tom_holland_brad_pitt_1': 'Tom Holland → Brad Pitt: one intermediary',
    'tom_holland_brad_pitt_3': 'Tom Holland → Brad Pitt: three intermediaries',
}
engines = ['sql', 'sparql'] if COMPARE_SQL else ['sparql']
queries = {}

### 2.1. Movies longer than 90 minutes

In [3]:
queries['movies_over_90'] = {
    "sql": """
SELECT title AS film, duration
FROM show
WHERE type_show = 'Movie' AND duration > 90
LIMIT 20;
""".strip() + "\n",
    "sparql": """
PREFIX show: <http://ontology.show-catalog.org/show#>

SELECT ?film ?duration
WHERE {
  ?movie a show:Movie ;
         show:name ?film ;
         show:duration ?duration .
  FILTER (?duration > 90)
}
LIMIT 20
""".strip() + "\n",
}

if COMPARE_SQL:
    display(Markdown("**SQL**"))
    display(Code(queries['movies_over_90']["sql"], language="sql"))
display(Markdown("**SPARQL**"))
display(Code(queries['movies_over_90']["sparql"], language="sparql"))

**SQL**

SELECT title AS film, duration
FROM show
WHERE type_show = 'Movie' AND duration > 90
LIMIT 20;

**SPARQL**

PREFIX show: <http://ontology.show-catalog.org/show#>

SELECT ?film ?duration
WHERE {
  ?movie a show:Movie ;
         show:name ?film ;
         show:duration ?duration .
  FILTER (?duration > 90)
}
LIMIT 20

### 2.2. Tom Holland → Brad Pitt: one intermediary

In [4]:
queries['tom_holland_brad_pitt_1'] = {
    "sql": """
SELECT
    p.firstname || ' ' || p.lastname AS intermediary1,
    f1.title AS film1,
    f2.title AS film2
FROM acted_in tom
JOIN acted_in bridge1 ON bridge1.show_id = tom.show_id
JOIN person p ON p.person_id = bridge1.person_id
JOIN acted_in bridge2 ON bridge2.person_id = p.person_id
JOIN acted_in brad ON brad.show_id = bridge2.show_id
JOIN show f1 ON f1.show_id = tom.show_id
JOIN show f2 ON f2.show_id = brad.show_id
WHERE tom.person_id = 6511
  AND brad.person_id = 2290
  AND p.person_id NOT IN (6511, 2290)
  AND f1.show_id <> f2.show_id
  AND f1.type_show = 'Movie'
  AND f2.type_show = 'Movie'
LIMIT 20;
""".strip() + "\n",
    "sparql": """
PREFIX show: <http://ontology.show-catalog.org/show#>
PREFIX person: <http://data.show-catalog.org/person/>

SELECT ?intermediary1 ?film1 ?film2
WHERE {
  person:6511 show:Acted_in ?f1 .
  ?actor show:Acted_in ?f1, ?f2 ; show:name ?intermediary1 .
  person:2290 show:Acted_in ?f2 .
  ?f1 a show:Movie ; show:name ?film1 .
  ?f2 a show:Movie ; show:name ?film2 .
  FILTER (?actor NOT IN (person:6511, person:2290))
  FILTER (?f1 != ?f2)
}
LIMIT 20
""".strip() + "\n",
}

if COMPARE_SQL:
    display(Markdown("**SQL**"))
    display(Code(queries['tom_holland_brad_pitt_1']["sql"], language="sql"))
display(Markdown("**SPARQL**"))
display(Code(queries['tom_holland_brad_pitt_1']["sparql"], language="sparql"))

**SQL**

SELECT
    p.firstname || ' ' || p.lastname AS intermediary1,
    f1.title AS film1,
    f2.title AS film2
FROM acted_in tom
JOIN acted_in bridge1 ON bridge1.show_id = tom.show_id
JOIN person p ON p.person_id = bridge1.person_id
JOIN acted_in bridge2 ON bridge2.person_id = p.person_id
JOIN acted_in brad ON brad.show_id = bridge2.show_id
JOIN show f1 ON f1.show_id = tom.show_id
JOIN show f2 ON f2.show_id = brad.show_id
WHERE tom.person_id = 6511
  AND brad.person_id = 2290
  AND p.person_id NOT IN (6511, 2290)
  AND f1.show_id <> f2.show_id
  AND f1.type_show = 'Movie'
  AND f2.type_show = 'Movie'
LIMIT 20;

**SPARQL**

PREFIX show: <http://ontology.show-catalog.org/show#>
PREFIX person: <http://data.show-catalog.org/person/>

SELECT ?intermediary1 ?film1 ?film2
WHERE {
  person:6511 show:Acted_in ?f1 .
  ?actor show:Acted_in ?f1, ?f2 ; show:name ?intermediary1 .
  person:2290 show:Acted_in ?f2 .
  ?f1 a show:Movie ; show:name ?film1 .
  ?f2 a show:Movie ; show:name ?film2 .
  FILTER (?actor NOT IN (person:6511, person:2290))
  FILTER (?f1 != ?f2)
}
LIMIT 20

### 2.3. Tom Holland -> Brad Pitt: three intermediaries

The SQL CTE builds actor collaborations; four joins of this CTE build the complete path.

In [5]:
queries['tom_holland_brad_pitt_3'] = {
    "sql": """
WITH collaborations AS (
    SELECT a.person_id AS source_actor, b.person_id AS target_actor,
           s.show_id AS film_id, s.title AS title
    FROM acted_in a
    JOIN acted_in b ON b.show_id = a.show_id
    JOIN show s ON s.show_id = a.show_id
    WHERE a.person_id <> b.person_id AND s.type_show = 'Movie'
)
SELECT
    p1.firstname || ' ' || p1.lastname AS intermediary1,
    p2.firstname || ' ' || p2.lastname AS intermediary2,
    p3.firstname || ' ' || p3.lastname AS intermediary3,
    c1.title AS film1, c2.title AS film2,
    c3.title AS film3, c4.title AS film4
FROM collaborations c1
JOIN collaborations c2 ON c2.source_actor = c1.target_actor
JOIN collaborations c3 ON c3.source_actor = c2.target_actor
JOIN collaborations c4 ON c4.source_actor = c3.target_actor
JOIN person p1 ON p1.person_id = c1.target_actor
JOIN person p2 ON p2.person_id = c2.target_actor
JOIN person p3 ON p3.person_id = c3.target_actor
WHERE c1.source_actor = 6511 AND c4.target_actor = 2290
  AND p1.person_id NOT IN (6511, 2290)
  AND p2.person_id NOT IN (6511, 2290)
  AND p3.person_id NOT IN (6511, 2290)
  AND p1.person_id <> p2.person_id
  AND p1.person_id <> p3.person_id
  AND p2.person_id <> p3.person_id
  AND c2.film_id <> c1.film_id
  AND c3.film_id NOT IN (c1.film_id, c2.film_id)
  AND c4.film_id NOT IN (c1.film_id, c2.film_id, c3.film_id)
LIMIT 20;
""".strip() + "\n",
    "sparql": """
PREFIX show: <http://ontology.show-catalog.org/show#>
PREFIX person: <http://data.show-catalog.org/person/>
SELECT ?intermediary1 ?intermediary2 ?intermediary3
       ?film1 ?film2 ?film3 ?film4
WHERE {
  person:6511 show:Acted_in ?f1 .
  ?a show:Acted_in ?f1, ?f2 ; show:name ?intermediary1 .
  ?b show:Acted_in ?f2, ?f3 ; show:name ?intermediary2 .
  ?c show:Acted_in ?f3, ?f4 ; show:name ?intermediary3 .
  person:2290 show:Acted_in ?f4 .
  ?f1 a show:Movie ; show:name ?film1 .
  ?f2 a show:Movie ; show:name ?film2 .
  ?f3 a show:Movie ; show:name ?film3 .
  ?f4 a show:Movie ; show:name ?film4 .
  FILTER (?a NOT IN (person:6511, person:2290))
  FILTER (?b NOT IN (person:6511, person:2290))
  FILTER (?c NOT IN (person:6511, person:2290))
  FILTER (?a != ?b && ?a != ?c && ?b != ?c)
  FILTER (?f1 != ?f2 && ?f1 != ?f3 && ?f1 != ?f4 &&
          ?f2 != ?f3 && ?f2 != ?f4 && ?f3 != ?f4)
}
LIMIT 20
""".strip() + "\n",
}

if COMPARE_SQL:
    display(Markdown("**SQL**"))
    display(Code(queries['tom_holland_brad_pitt_3']["sql"], language="sql"))
display(Markdown("**SPARQL**"))
display(Code(queries['tom_holland_brad_pitt_3']["sparql"], language="sparql"))

**SQL**

WITH collaborations AS (
    SELECT a.person_id AS source_actor, b.person_id AS target_actor,
           s.show_id AS film_id, s.title AS title
    FROM acted_in a
    JOIN acted_in b ON b.show_id = a.show_id
    JOIN show s ON s.show_id = a.show_id
    WHERE a.person_id <> b.person_id AND s.type_show = 'Movie'
)
SELECT
    p1.firstname || ' ' || p1.lastname AS intermediary1,
    p2.firstname || ' ' || p2.lastname AS intermediary2,
    p3.firstname || ' ' || p3.lastname AS intermediary3,
    c1.title AS film1, c2.title AS film2,
    c3.title AS film3, c4.title AS film4
FROM collaborations c1
JOIN collaborations c2 ON c2.source_actor = c1.target_actor
JOIN collaborations c3 ON c3.source_actor = c2.target_actor
JOIN collaborations c4 ON c4.source_actor = c3.target_actor
JOIN person p1 ON p1.person_id = c1.target_actor
JOIN person p2 ON p2.person_id = c2.target_actor
JOIN person p3 ON p3.person_id = c3.target_actor
WHERE c1.source_actor = 6511 AND c4.target_actor = 2290
  AND p1.person_id NOT IN (6511, 2290)
  AND p2.person_id NOT IN (6511, 2290)
  AND p3.person_id NOT IN (6511, 2290)
  AND p1.person_id <> p2.person_id
  AND p1.person_id <> p3.person_id
  AND p2.person_id <> p3.person_id
  AND c2.film_id <> c1.film_id
  AND c3.film_id NOT IN (c1.film_id, c2.film_id)
  AND c4.film_id NOT IN (c1.film_id, c2.film_id, c3.film_id)
LIMIT 20;

**SPARQL**

PREFIX show: <http://ontology.show-catalog.org/show#>
PREFIX person: <http://data.show-catalog.org/person/>
SELECT ?intermediary1 ?intermediary2 ?intermediary3
       ?film1 ?film2 ?film3 ?film4
WHERE {
  person:6511 show:Acted_in ?f1 .
  ?a show:Acted_in ?f1, ?f2 ; show:name ?intermediary1 .
  ?b show:Acted_in ?f2, ?f3 ; show:name ?intermediary2 .
  ?c show:Acted_in ?f3, ?f4 ; show:name ?intermediary3 .
  person:2290 show:Acted_in ?f4 .
  ?f1 a show:Movie ; show:name ?film1 .
  ?f2 a show:Movie ; show:name ?film2 .
  ?f3 a show:Movie ; show:name ?film3 .
  ?f4 a show:Movie ; show:name ?film4 .
  FILTER (?a NOT IN (person:6511, person:2290))
  FILTER (?b NOT IN (person:6511, person:2290))
  FILTER (?c NOT IN (person:6511, person:2290))
  FILTER (?a != ?b && ?a != ?c && ?b != ?c)
  FILTER (?f1 != ?f2 && ?f1 != ?f3 && ?f1 != ?f4 &&
          ?f2 != ?f3 && ?f2 != ?f4 && ?f3 != ?f4)
}
LIMIT 20

## 3. Independent validation against the CSVs

Every result is checked **outside the timed section**: a correct movie duration,
or distinct actors and films with every cast membership present in the CSVs.
Validation uses identifiers, even when two titles or names are identical.

In [6]:
def load_source():
    data = HERE.parent / 'submit' / 'Database'

    def read(name):
        with (data / f'{name}.csv').open(encoding='utf-8-sig', newline='') as stream:
            return list(csv.DictReader(stream))

    people, movies, cast = defaultdict(set), defaultdict(set), defaultdict(set)
    for row in read('person'):
        people[row['firstname'] + ' ' + row['lastname']].add(row['no_id'])
    durations = defaultdict(set)
    for row in read('show'):
        if row['type'] == 'Movie':
            movies[row['title']].add(row['show_id'])
            durations[row['title']].add(int(row['duration']))
    for row in read('actor'):
        cast[row['show_id']].add(row['cast_id'])
    return people, movies, cast, durations

In [7]:
def validate(columns, rows, source):
    """Verify distinct actors/films and every cast edge, outside the timed section."""
    if not 0 < len(rows) <= 20:
        raise ValueError(f'Expected 1 to 20 rows, got {len(rows)}')
    people, movies, cast, durations = source
    for values in rows:
        row = dict(zip(columns, values))
        if 'duration' in row:
            duration = int(row['duration'])
            if duration <= 90 or duration not in durations[row['film']]:
                raise ValueError(f'Invalid movie/duration: {row}')
            continue
        intermediaries = [row[c] for c in columns if c.startswith('intermediary')]
        films = [row[c] for c in columns if c.startswith('film')]
        actors = [people[name] for name in intermediaries] + [{'2290'}]
        if len(films) != len(actors):
            raise ValueError('A path needs one film per collaboration')
        states = [(('6511',), ())]
        for index, title in enumerate(films):
            next_states = []
            for used_people, used_movies in states:
                for movie in movies[title] - set(used_movies):
                    if used_people[-1] in cast[movie]:
                        for person in (actors[index] & cast[movie]) - set(used_people):
                            next_states.append((used_people + (person,), used_movies + (movie,)))
            states = next_states
        if not states:
            raise ValueError(f'Invalid actor/film path: {row}')

In [8]:
source = load_source()
path_columns = ['intermediary1', 'intermediary2', 'intermediary3',
                'film1', 'film2', 'film3', 'film4']
example_path = ['Ewan McGregor', 'Abigail Breslin', 'Catherine Zeta-Jones',
                'The Impossible', 'August: Osage County', 'No Reservations', "Ocean's Twelve"]
validate(path_columns, [example_path], source)

# Small regression checks: reject a missing collaboration and a repeated movie.
for index, replacement in [(0, 'Don Cheadle'), (5, 'The Impossible')]:
    invalid_path = example_path.copy()
    invalid_path[index] = replacement
    try:
        validate(path_columns, [invalid_path], source)
    except ValueError:
        pass
    else:
        raise AssertionError('An invalid path was accepted')
try:
    validate(['film', 'duration'], [('The Impossible', 90)], source)
except ValueError:
    pass
else:
    raise AssertionError('The duration filter was ignored')
print('Presentation path validated; three negative checks passed.')

Presentation path validated; three negative checks passed.


## 4. Performance measurements

Queries run sequentially, alternating SQL/SPARQL order between repetitions.
Client elapsed time includes execution, transfer, and response decoding.
PostgreSQL is queried in read-only mode with a 45-second statement timeout;
the HTTP connection to Fuseki has a 45-second read timeout.

In [9]:
endpoint = urlsplit(SPARQL_ENDPOINT)
assert endpoint.scheme in ('http', 'https') and endpoint.hostname
assert not endpoint.query and not endpoint.fragment
http_type = http.client.HTTPSConnection if endpoint.scheme == 'https' else http.client.HTTPConnection
http_conn = http_type(endpoint.hostname, endpoint.port, timeout=45)
db = None

def run_sparql(query):
    http_conn.request('POST', endpoint.path or '/', urlencode({'query': query}),{'Content-Type': 'application/x-www-form-urlencoded','Accept': 'application/sparql-results+json'})
    response = http_conn.getresponse()
    raw = response.read()
    if response.status != 200:
        raise RuntimeError(f'HTTP {response.status}: {raw[:500]!r}')
    payload = json.loads(raw)
    columns = payload['head']['vars']
    rows = [tuple(row[col]['value'] for col in columns)
            for row in payload['results']['bindings']]
    return columns, rows

def run_sql(query):
    with db.cursor() as cursor:
        cursor.execute(query)
        return [col.name for col in cursor.description], cursor.fetchall()

try:
    http_conn.connect()
    postgres_version = None
    if COMPARE_SQL:
        import psycopg2
        db = psycopg2.connect(**PG_CONFIG, connect_timeout=5,
            options='-c statement_timeout=45000 -c default_transaction_read_only=on')
        db.autocommit = True
        with db.cursor() as cursor:
            cursor.execute('SELECT version()')
            postgres_version = cursor.fetchone()[0]
    print('Connections established. Ready to measure queries.')
except Exception:
    http_conn.close()
    if db is not None:
        db.close()
    raise

Connections established. Ready to measure queries.


In [10]:
started = datetime.now(timezone.utc)
report = {
    'utc': started.isoformat(), 'python': platform.python_version(),
    'postgres_version': postgres_version, 'endpoint': SPARQL_ENDPOINT,
    'repeats': REPEATS, 'limit': 20,
    'method': 'Client elapsed including transfer and decoding; established connections; '
              'sequential queries with alternating engines; caches not reset.',
    'cases': {},
}
try:
    for case in CASES:
        report['cases'][case] = {
            engine: {'query_sha256': hashlib.sha256(queries[case][engine].encode()).hexdigest(),
                     'runs': []}
            for engine in engines
        }
        for iteration in range(REPEATS + 1):
            for engine in (engines if iteration % 2 == 0 else engines[::-1]):
                start = time.perf_counter()
                columns, rows = (run_sql if engine == 'sql' else run_sparql)(queries[case][engine])
                elapsed = round((time.perf_counter() - start) * 1000, 3)
                validate(columns, rows, source)
                result = report['cases'][case][engine]
                result['runs'].append({'iteration': iteration, 'ms': elapsed,'rows': len(rows), 'all_rows_validated': True})
                result['sample_results'] = [dict(zip(columns, row)) for row in rows]
        print(f"{CASES[case]} : {REPEATS + 1} executions per engine, results validated.")
except Exception:
    http_conn.close()
    if db is not None:
        db.close()
    raise

Movies longer than 90 minutes : 6 executions per engine, results validated.


Tom Holland → Brad Pitt: one intermediary : 6 executions per engine, results validated.


Tom Holland → Brad Pitt: three intermediaries : 6 executions per engine, results validated.


## 5. Results and examples

The median includes only repetitions **after** the first measured execution.
The first measurement is not a cold-cache test: the services and caches may
already have been used.

In [11]:
def markdown_table(columns, rows):
    def escape(value):
        return str(value).replace('|', r'\|').replace('\n', ' ')
    return '\n'.join([
        '| ' + ' | '.join(map(escape, columns)) + ' |',
        '| ' + ' | '.join(['---'] * len(columns)) + ' |',
        *['| ' + ' | '.join(map(escape, row)) + ' |' for row in rows],
    ])

summary_rows = []
for case, results in report['cases'].items():
    for engine, result in results.items():
        runs = result['runs']
        warm = [run['ms'] for run in runs if run['iteration'] > 0]
        result['summary'] = {
            'first_ms': runs[0]['ms'], 'repetitions': len(warm),
            'median_ms': round(statistics.median(warm), 3),
            'min_ms': min(warm), 'max_ms': max(warm),
        }
        s = result['summary']
        summary_rows.append([CASES[case], engine.upper(), s['first_ms'], s['median_ms'],f"{s['min_ms']}–{s['max_ms']}", runs[-1]['rows']])
summary_markdown = markdown_table(
    ['Query', 'Engine', 'First (ms)', 'Median (ms)', 'Min–max (ms)', 'Rows'], summary_rows)
display(Markdown(summary_markdown))

if COMPARE_SQL:
    case = report['cases']['tom_holland_brad_pitt_3']
    ratio = case['sql']['summary']['median_ms'] / case['sparql']['summary']['median_ms']
    print(f'SQL / SPARQL median ratio for three intermediaries: {ratio:.2f}')

| Query | Engine | First (ms) | Median (ms) | Min–max (ms) | Rows |
| --- | --- | --- | --- | --- | --- |
| Movies longer than 90 minutes | SQL | 12.528 | 0.768 | 0.541–0.84 | 20 |
| Movies longer than 90 minutes | SPARQL | 27.262 | 5.577 | 4.703–6.989 | 20 |
| Tom Holland → Brad Pitt: one intermediary | SQL | 27.393 | 14.528 | 13.797–16.359 | 2 |
| Tom Holland → Brad Pitt: one intermediary | SPARQL | 8.317 | 6.666 | 4.46–7.713 | 2 |
| Tom Holland → Brad Pitt: three intermediaries | SQL | 190.812 | 165.724 | 163.78–186.77 | 20 |
| Tom Holland → Brad Pitt: three intermediaries | SPARQL | 39.444 | 18.123 | 15.306–24.746 | 20 |

SQL / SPARQL median ratio for three intermediaries: 9.14


In [12]:
display(Markdown('**One intermediary: collaborations through Don Cheadle**'))
samples = report['cases']['tom_holland_brad_pitt_1']['sparql']['sample_results']
display(Markdown(markdown_table(['Intermediary', 'Film with Tom Holland', 'Film with Brad Pitt'],
    [[row['intermediary1'], row['film1'], row['film2']] for row in samples])))

if COMPARE_SQL:
    results = report['cases']['tom_holland_brad_pitt_1']
    sql_rows = {tuple(row[col] for col in ['intermediary1', 'film1', 'film2'])
                for row in results['sql']['sample_results']}
    sparql_rows = {tuple(row[col] for col in ['intermediary1', 'film1', 'film2'])
                   for row in results['sparql']['sample_results']}
    assert sql_rows == sparql_rows, 'The engines returned different one-intermediary results'
    print('Both engines returned the same one-intermediary collaborations.')

**One intermediary: collaborations through Don Cheadle**

| Intermediary | Film with Tom Holland | Film with Brad Pitt |
| --- | --- | --- |
| Don Cheadle | Avengers: Infinity War | Ocean's Thirteen |
| Don Cheadle | Avengers: Infinity War | Ocean's Twelve |

Both engines returned the same one-intermediary collaborations.


## 6. Presentation path: three intermediaries

**Tom Holland → The Impossible → Ewan McGregor → August: Osage County →
Abigail Breslin → No Reservations → Catherine Zeta-Jones → Ocean's Twelve → Brad Pitt**

The following query pins the intermediaries using `VALUES` to retrieve this example,
even when it does not appear among the first 20 results of the general search.
This targeted query **is not included in the benchmark timings**.

In [13]:
example_query = """
PREFIX show: <http://ontology.show-catalog.org/show#>
PREFIX person: <http://data.show-catalog.org/person/>

# Pin the three intermediaries so LIMIT 20 cannot hide the presentation example.
SELECT ?intermediary1 ?intermediary2 ?intermediary3
       ?film1 ?film2 ?film3 ?film4
WHERE {
  VALUES (?a ?b ?c) { (person:3107 person:3461 person:5303) }
  person:6511 show:Acted_in ?f1 .
  ?a show:Acted_in ?f1, ?f2 ; show:name ?intermediary1 .
  ?b show:Acted_in ?f2, ?f3 ; show:name ?intermediary2 .
  ?c show:Acted_in ?f3, ?f4 ; show:name ?intermediary3 .
  person:2290 show:Acted_in ?f4 .
  ?f1 a show:Movie ; show:name ?film1 .
  ?f2 a show:Movie ; show:name ?film2 .
  ?f3 a show:Movie ; show:name ?film3 .
  ?f4 a show:Movie ; show:name ?film4 .
  FILTER (?f1 != ?f2 && ?f1 != ?f3 && ?f1 != ?f4 &&
          ?f2 != ?f3 && ?f2 != ?f4 && ?f3 != ?f4)
}
LIMIT 20
""".strip() + "\n"
display(Code(example_query, language='sparql'))
try:
    columns, rows = run_sparql(example_query)
    validate(columns, rows, source)
    report['example'] = [dict(zip(columns, row)) for row in rows]
    assert any([row[col] for col in path_columns] == example_path for row in report['example'])
    display(Markdown(markdown_table(path_columns, [[row[col] for col in path_columns]for row in report['example']])))
    print('Path retrieved from Fuseki and verified against the source CSVs.')
finally:
    http_conn.close()
    if db is not None:
        db.close()

PREFIX show: <http://ontology.show-catalog.org/show#>
PREFIX person: <http://data.show-catalog.org/person/>

# Pin the three intermediaries so LIMIT 20 cannot hide the presentation example.
SELECT ?intermediary1 ?intermediary2 ?intermediary3
       ?film1 ?film2 ?film3 ?film4
WHERE {
  VALUES (?a ?b ?c) { (person:3107 person:3461 person:5303) }
  person:6511 show:Acted_in ?f1 .
  ?a show:Acted_in ?f1, ?f2 ; show:name ?intermediary1 .
  ?b show:Acted_in ?f2, ?f3 ; show:name ?intermediary2 .
  ?c show:Acted_in ?f3, ?f4 ; show:name ?intermediary3 .
  person:2290 show:Acted_in ?f4 .
  ?f1 a show:Movie ; show:name ?film1 .
  ?f2 a show:Movie ; show:name ?film2 .
  ?f3 a show:Movie ; show:name ?film3 .
  ?f4 a show:Movie ; show:name ?film4 .
  FILTER (?f1 != ?f2 && ?f1 != ?f3 && ?f1 != ?f4 &&
          ?f2 != ?f3 && ?f2 != ?f4 && ?f3 != ?f4)
}
LIMIT 20

| intermediary1 | intermediary2 | intermediary3 | film1 | film2 | film3 | film4 |
| --- | --- | --- | --- | --- | --- | --- |
| Ewan McGregor | Abigail Breslin | Catherine Zeta-Jones | The Impossible | August: Osage County | No Reservations | Ocean's Twelve |

Path retrieved from Fuseki and verified against the source CSVs.


## 7. Comparison limits

Measurements and examples are displayed directly in this notebook.

- `LIMIT 20` without `ORDER BY`: the engines may select different paths.
  The test measures the time to retrieve up to 20 answers, without enumerating all paths.
- The three-intermediary SQL keeps the CTE from the first test. Different SQL plans,
  indexes, or configurations may change the performance results.
- The protocols and engines differ: PostgreSQL uses its driver, while Fuseki uses HTTP.
  The figures include client overhead; they are not pure CPU timings.
- This three-intermediary path is not the shortest: Don Cheadle already provides a shorter connection.
- These queries traverse relationships; they do not enable OWL reasoning.